# World Cup Match Outcome Predictor

## Commit 4: Leakage-safe rolling team form

This notebook stage rebuilds recent-team-form features using the deterministic `match_id` introduced in Commit 3. Each historical match contributes exactly one home and one away team-perspective row. Form is calculated from the previous five matches strictly before the current match date, so matches recorded on the same date cannot leak results into one another. Home and away features are then joined back to the match table with one-to-one `match_id` merges.


In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

# Resolve the project root whether Jupyter starts in the repository root
# or inside the notebooks directory.
current_dir = Path.cwd()
if (current_dir / "data" / "results.csv").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "results.csv").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not locate data/results.csv. Start Jupyter from the project folder "
        "or keep the repository structure unchanged."
    )

results_path = project_root / "data" / "results.csv"
outputs_dir = project_root / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

REQUIRED_COLUMNS = [
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament",
    "city",
    "country",
    "neutral",
]

# Load the source data without silently assuming the expected schema.
df_raw = pd.read_csv(results_path)
raw_row_count = len(df_raw)

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df_raw.columns))
if missing_columns:
    raise ValueError(
        "results.csv is missing required columns: "
        + ", ".join(missing_columns)
    )

# Retain only the documented source fields at this stage.
df_raw = df_raw[REQUIRED_COLUMNS].copy()

# Parse dates and scores consistently. Invalid values become missing and are
# counted before incomplete matches are removed.
df_raw["date"] = pd.to_datetime(df_raw["date"], errors="coerce")
df_raw["home_score"] = pd.to_numeric(df_raw["home_score"], errors="coerce")
df_raw["away_score"] = pd.to_numeric(df_raw["away_score"], errors="coerce")

invalid_date_rows = int(df_raw["date"].isna().sum())
missing_score_rows = int(
    df_raw[["home_score", "away_score"]].isna().any(axis=1).sum()
)
exact_duplicate_rows = int(
    df_raw.duplicated(subset=REQUIRED_COLUMNS, keep="first").sum()
)

match_key_columns = ["date", "home_team", "away_team"]
duplicate_match_mask = df_raw.duplicated(
    subset=match_key_columns, keep=False
)
duplicate_match_key_rows = int(duplicate_match_mask.sum())
duplicate_match_key_groups = int(
    df_raw.loc[duplicate_match_mask, match_key_columns]
    .drop_duplicates()
    .shape[0]
)

# Save the potential duplicate-key rows for manual review. They are not
# automatically removed because two records can share teams and a date while
# still representing distinct source observations.
duplicate_report_path = outputs_dir / "duplicate_match_keys.csv"
df_raw.loc[duplicate_match_mask].sort_values(match_key_columns).to_csv(
    duplicate_report_path, index=False
)

# Remove records that cannot be used as completed match outcomes and remove
# exact duplicate rows only.
df_matches = (
    df_raw.dropna(subset=["date", "home_score", "away_score"])
    .drop_duplicates(subset=REQUIRED_COLUMNS, keep="first")
    .copy()
)

# Create a stable identifier from the complete validated source record.
# SHA-1 is used only as a deterministic identifier, not for security.
def make_match_id(row):
    values = []
    for column in REQUIRED_COLUMNS:
        value = row[column]
        if column == "date":
            value = value.strftime("%Y-%m-%d")
        elif isinstance(value, float) and value.is_integer():
            value = int(value)
        values.append(str(value).strip())
    payload = "|".join(values)
    return "match_" + hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]

df_matches["match_id"] = df_matches.apply(make_match_id, axis=1)

if not df_matches["match_id"].is_unique:
    raise ValueError(
        "Generated match_id values are not unique. Review exact duplicates "
        "or extend the identifier fields."
    )

# Keep a predictable column order for later feature-engineering commits.
df_matches = (
    df_matches[["match_id"] + REQUIRED_COLUMNS]
    .sort_values(["date", "home_team", "away_team", "match_id"])
    .reset_index(drop=True)
)

validation_summary = pd.DataFrame(
    [
        ("source_rows", raw_row_count),
        ("required_columns", len(REQUIRED_COLUMNS)),
        ("missing_required_columns", len(missing_columns)),
        ("invalid_date_rows", invalid_date_rows),
        ("missing_score_rows", missing_score_rows),
        ("exact_duplicate_rows_removed", exact_duplicate_rows),
        ("duplicate_match_key_groups_reported", duplicate_match_key_groups),
        ("duplicate_match_key_rows_reported", duplicate_match_key_rows),
        ("validated_completed_matches", len(df_matches)),
        ("unique_match_ids", df_matches["match_id"].nunique()),
        ("earliest_match_date", df_matches["date"].min().date().isoformat()),
        ("latest_match_date", df_matches["date"].max().date().isoformat()),
    ],
    columns=["metric", "value"],
)

validation_summary_path = outputs_dir / "data_validation_summary.csv"
validation_summary.to_csv(validation_summary_path, index=False)

print("Project root:", project_root)
print("Dataset path:", results_path)
print("Validated dataset shape:", df_matches.shape)
print("Validation summary:", validation_summary_path)
print("Duplicate-key report:", duplicate_report_path)
display(validation_summary)
display(df_matches.head())


In [ ]:
# Filter to the modern modelling period after validation.
modern_era_cutoff = pd.Timestamp("2000-01-01")
df_clean = df_matches.loc[df_matches["date"] >= modern_era_cutoff].copy()

if not df_clean["match_id"].is_unique:
    raise ValueError("match_id must remain unique after filtering.")

print("Validated completed matches:", len(df_matches))
print("Total matches from 2000 onward:", len(df_clean))
print("Unique modern-era match IDs:", df_clean["match_id"].nunique())
print("\nTop 10 tournament types:")
print(df_clean["tournament"].value_counts().head(10))


In [ ]:
# Build one team-perspective row for each side of every validated match.
# match_id is retained so form can be joined back without ambiguous date/team merges.
df_home = df_clean[[
    'match_id', 'date', 'home_team', 'home_score', 'away_score'
]].copy()
df_home['side'] = 'home'
df_home = df_home.rename(columns={
    'home_team': 'team',
    'home_score': 'goals_for',
    'away_score': 'goals_against'
})

df_away = df_clean[[
    'match_id', 'date', 'away_team', 'away_score', 'home_score'
]].copy()
df_away['side'] = 'away'
df_away = df_away.rename(columns={
    'away_team': 'team',
    'away_score': 'goals_for',
    'home_score': 'goals_against'
})

df_team_matches = (
    pd.concat([df_home, df_away], ignore_index=True)
    .sort_values(['team', 'date', 'match_id', 'side'])
    .reset_index(drop=True)
)

expected_team_rows = 2 * len(df_clean)
if len(df_team_matches) != expected_team_rows:
    raise ValueError(
        f'Expected {expected_team_rows} team-perspective rows, '
        f'found {len(df_team_matches)}.'
    )

# Rolling averages use shift(1), so the current result cannot enter its own
# feature values. The window contains the previous five individual matches.
window_size = 5
df_team_matches['form_goals_for'] = (
    df_team_matches.groupby('team')['goals_for']
    .transform(lambda s: s.shift(1).rolling(window_size, min_periods=1).mean())
)
df_team_matches['form_goals_against'] = (
    df_team_matches.groupby('team')['goals_against']
    .transform(lambda s: s.shift(1).rolling(window_size, min_periods=1).mean())
)

# The source contains a handful of cases where one team has multiple matches
# on the same calendar date. Because no kick-off times are available, all
# matches on that date must use the same form calculated before that date.
# Taking the first shifted value for each team/date group prevents same-day leakage.
df_team_matches['form_goals_for'] = (
    df_team_matches.groupby(['team', 'date'])['form_goals_for'].transform('first')
)
df_team_matches['form_goals_against'] = (
    df_team_matches.groupby(['team', 'date'])['form_goals_against'].transform('first')
)

# A team's first observed match has no prior history, so use a neutral zero
# rather than borrowing any future information.
df_team_matches[['form_goals_for', 'form_goals_against']] = (
    df_team_matches[['form_goals_for', 'form_goals_against']].fillna(0.0)
)

same_day_rows = df_team_matches[
    df_team_matches.duplicated(['team', 'date'], keep=False)
]
same_day_form_counts = same_day_rows.groupby(['team', 'date'])[
    ['form_goals_for', 'form_goals_against']
].nunique()
if not same_day_form_counts.empty and (same_day_form_counts > 1).any().any():
    raise ValueError('Same-day matches received different pre-date form values.')

print('Team-perspective rows:', len(df_team_matches))
print('Expected team-perspective rows:', expected_team_rows)
print('Same-day team/date groups handled:', len(same_day_form_counts))
print("\nArgentina's leakage-safe form check:")
display(
    df_team_matches.loc[df_team_matches['team'].eq('Argentina'), [
        'match_id', 'date', 'side', 'team', 'goals_for', 'goals_against',
        'form_goals_for', 'form_goals_against'
    ]].tail(10)
)


In [ ]:
# Create one unambiguous form lookup for each side of every match.
home_form = (
    df_team_matches.loc[
        df_team_matches['side'].eq('home'),
        ['match_id', 'form_goals_for', 'form_goals_against']
    ]
    .rename(columns={
        'form_goals_for': 'home_form_goals_for',
        'form_goals_against': 'home_form_goals_against'
    })
)

away_form = (
    df_team_matches.loc[
        df_team_matches['side'].eq('away'),
        ['match_id', 'form_goals_for', 'form_goals_against']
    ]
    .rename(columns={
        'form_goals_for': 'away_form_goals_for',
        'form_goals_against': 'away_form_goals_against'
    })
)

if not home_form['match_id'].is_unique:
    raise ValueError('Home form lookup contains duplicate match_id values.')
if not away_form['match_id'].is_unique:
    raise ValueError('Away form lookup contains duplicate match_id values.')

# One-to-one validation makes accidental row multiplication fail loudly.
df_model = (
    df_clean
    .merge(home_form, on='match_id', how='left', validate='one_to_one')
    .merge(away_form, on='match_id', how='left', validate='one_to_one')
)

form_columns = [
    'home_form_goals_for', 'home_form_goals_against',
    'away_form_goals_for', 'away_form_goals_against'
]
if df_model[form_columns].isna().any().any():
    raise ValueError('Missing rolling-form values after match_id merge.')

if len(df_model) != len(df_clean):
    raise ValueError(
        f'Feature merge changed the match count from {len(df_clean)} '
        f'to {len(df_model)}.'
    )
if not df_model['match_id'].is_unique:
    raise ValueError('df_model must contain exactly one row per match_id.')

# Target encoding: 2 = Home Win, 1 = Draw, 0 = Away Win.
conditions = [
    df_model['home_score'] > df_model['away_score'],
    df_model['home_score'] == df_model['away_score'],
    df_model['home_score'] < df_model['away_score'],
]
choices = [2, 1, 0]
df_model['target'] = np.select(conditions, choices, default=np.nan)

if df_model['target'].isna().any():
    raise ValueError('Target encoding produced missing values.')

# Write a compact machine-readable check for this commit.
rolling_form_summary = pd.DataFrame(
    [
        ('modern_matches', len(df_clean)),
        ('team_perspective_rows', len(df_team_matches)),
        ('expected_team_perspective_rows', 2 * len(df_clean)),
        ('same_day_team_date_groups', len(same_day_form_counts)),
        ('home_form_lookup_rows', len(home_form)),
        ('away_form_lookup_rows', len(away_form)),
        ('final_model_rows', len(df_model)),
        ('unique_model_match_ids', df_model['match_id'].nunique()),
        ('duplicate_model_match_ids', int(df_model['match_id'].duplicated().sum())),
        ('rows_added_or_lost', len(df_model) - len(df_clean)),
        ('missing_form_values', int(df_model[form_columns].isna().sum().sum())),
    ],
    columns=['metric', 'value']
)

rolling_form_summary_path = outputs_dir / 'rolling_form_validation.csv'
rolling_form_summary.to_csv(rolling_form_summary_path, index=False)

print('Final feature matrix rows:', len(df_model))
print('Unique match IDs:', df_model['match_id'].nunique())
print('Rows added or lost:', len(df_model) - len(df_clean))
print('Rolling-form validation:', rolling_form_summary_path)
display(rolling_form_summary)
display(df_model[[
    'match_id', 'date', 'home_team', 'away_team',
    'home_form_goals_for', 'home_form_goals_against',
    'away_form_goals_for', 'away_form_goals_against', 'target'
]].tail())


In [ ]:
import numpy as np
import pandas as pd

# Preserve the original prototype tournament-weight rules in this commit.
# Their logic will be reviewed separately in the next modelling commit.
def get_tournament_weight(tournament_name):
    if 'FIFA World Cup' in tournament_name and 'qualification' not in tournament_name:
        return 1.0
    elif 'Confederations Cup' in tournament_name or 'Copa America' in tournament_name or 'Euro' in tournament_name and 'qualification' not in tournament_name:
        return 0.8
    elif 'qualification' in tournament_name:
        return 0.6
    elif 'Nations League' in tournament_name:
        return 0.5
    elif 'Friendly' in tournament_name:
        return 0.25
    else:
        return 0.4

# df_model already contains exactly one row per match_id after the Commit 4
# rolling-form merge, so add context features directly instead of rebuilding
# the model table with the old ambiguous date/team join.
df_model['match_weight'] = df_model['tournament'].apply(get_tournament_weight)
df_model['is_neutral'] = df_model['neutral'].astype(int)

features = [
    'home_form_goals_for', 'home_form_goals_against',
    'away_form_goals_for', 'away_form_goals_against',
    'match_weight', 'is_neutral'
]
X = df_model[features]
y = df_model['target']

if len(X) != len(df_clean) or not df_model['match_id'].is_unique:
    raise ValueError('Context-feature step changed the one-row-per-match structure.')

print('Context columns added without rebuilding the rolling-form merge.')
print('Shape of Feature Matrix (X):', X.shape)
print('Shape of Target Vector (y):', y.shape)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# 1. Define our Feature Matrix (X) and Target Vector (y)
# We isolate only the predictive numeric columns, dropping metadata like dates and names
features = [
    'home_form_goals_for', 'home_form_goals_against',
    'away_form_goals_for', 'away_form_goals_against',
    'match_weight', 'is_neutral'
]

X = df_model[features]
y = df_model['target']

# 2. Split the pitch: 80% for training, 20% for testing our predictions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Standardize the data so the SVM hyperplanes aren't distorted
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Initialize and train the model WITH balanced class weights
print("Retraining the SVM with balanced weights...")
svm_model_balanced = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
svm_model_balanced.fit(X_train_scaled, y_train)

# 5. Evaluate the newly balanced model
y_pred_balanced = svm_model_balanced.predict(X_test_scaled)
print("\n--- Balanced Model Evaluation ---")
print(classification_report(y_test, y_pred_balanced, target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)']))

In [ ]:
# 1. Define standard Elo mathematical functions
def get_expected_score(rating_a, rating_b):
    # Calculates the probability of Team A winning based on Elo difference
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(rating, expected, actual, k=30):
    # k is the weight of the match.
    return rating + k * (actual - expected)

# 2. Initialize a dictionary to track everyone's live rating (Baseline is 1500)
current_elo = {}

home_elos = []
away_elos = []

# 3. Loop through history chronologically to calculate pre-match Elo for every game
print("Calculating historical Elo ratings...")
for index, row in df_clean.sort_values('date').iterrows():
    home = row['home_team']
    away = row['away_team']

    # If a team is new to the dataset, give them the baseline 1500 rating
    if home not in current_elo: current_elo[home] = 1500
    if away not in current_elo: current_elo[away] = 1500

    # Store the pre-match ratings to use as features later
    home_elos.append(current_elo[home])
    away_elos.append(current_elo[away])

    # Determine the actual outcome for the Elo math (1 = Home Win, 0.5 = Draw, 0 = Away Win)
    if row['home_score'] > row['away_score']:
        actual_home, actual_away = 1.0, 0.0
    elif row['home_score'] < row['away_score']:
        actual_home, actual_away = 0.0, 1.0
    else:
        actual_home, actual_away = 0.5, 0.5

    # Calculate expected outcomes
    expected_home = get_expected_score(current_elo[home], current_elo[away])
    expected_away = get_expected_score(current_elo[away], current_elo[home])

    # Update their live ratings using the match weight we created earlier
    k_adjusted = 30 * row['match_weight']
    current_elo[home] = update_elo(current_elo[home], expected_home, actual_home, k_adjusted)
    current_elo[away] = update_elo(current_elo[away], expected_away, actual_away, k_adjusted)

# 4. Attach these shiny new features to our clean dataset
df_clean_sorted = df_clean.sort_values('date').copy()
df_clean_sorted['home_elo'] = home_elos
df_clean_sorted['away_elo'] = away_elos

# Let's see who the top 5 teams are at the end of our dataset
print("\nTop 5 Teams by Final Elo Rating:")
top_teams = sorted(current_elo.items(), key=lambda x: x[1], reverse=True)[:5]
for team, rating in top_teams:
    print(f"{team}: {rating:.0f}")

In [ ]:
# 1. Merge the new Elo ratings into our existing model dataframe
df_model = pd.merge(
    df_model,
    df_clean_sorted[['date', 'home_team', 'away_team', 'home_elo', 'away_elo']],
    on=['date', 'home_team', 'away_team'],
    how='left'
)

# Clean up any potential NaNs from the merge
df_model = df_model.dropna()

# 2. Define our Upgraded Feature Matrix (X)
features_upgraded = [
    'home_elo', 'away_elo', # <-- The new heavy hitters
    'home_form_goals_for', 'home_form_goals_against',
    'away_form_goals_for', 'away_form_goals_against',
    'match_weight', 'is_neutral'
]

X_upgraded = df_model[features_upgraded]
y_upgraded = df_model['target']

# 3. Split and Scale the new pitch
X_train_up, X_test_up, y_train_up, y_test_up = train_test_split(X_upgraded, y_upgraded, test_size=0.2, random_state=42)

scaler_upgraded = StandardScaler()
X_train_scaled_up = scaler_upgraded.fit_transform(X_train_up)
X_test_scaled_up = scaler_upgraded.transform(X_test_up)

# 4. Retrain the SVM Engine
print("Retraining the SVM with Historical Elo Ratings...")
svm_model_final = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
svm_model_final.fit(X_train_scaled_up, y_train_up)

# 5. Evaluate the Ultimate Model
y_pred_final = svm_model_final.predict(X_test_scaled_up)
print("\n--- Final Model Evaluation (Elo + Form) ---")
print(classification_report(y_test_up, y_pred_final, target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)']))

In [ ]:
import numpy as np
from collections import Counter

# 1. The Match Prediction Engine
def get_match_probabilities(team_home, team_away):
    # In a real app, you would dynamically pull their live Elo and Form here.
    # For this simulation, we will construct a dummy feature vector representing a tight match.
    # Features: [home_elo, away_elo, home_gf, home_ga, away_gf, away_ga, weight, neutral]
    # Let's pretend they are evenly matched (Elo 1800) on neutral ground (1.0 weight, 1 neutral)
    dummy_features = [[1800, 1750, 2.0, 0.5, 1.8, 0.8, 1.0, 1]]

    # Scale and predict
    scaled_features = scaler_upgraded.transform(dummy_features)
    probs = svm_model_final.predict_proba(scaled_features)[0]

    # probs output is [P(Away Win), P(Draw), P(Home Win)]
    return probs[0], probs[1], probs[2]

# 2. The Single Match Simulator
def simulate_knockout_match(team_a, team_b):
    p_away, p_draw, p_home = get_match_probabilities(team_a, team_b)

    # In knockout football, there are no draws. If the SVM predicts a draw,
    # we simulate extra time/penalties by essentially tossing a coin (50/50).
    outcomes = [team_b, 'Draw', team_a]
    result = np.random.choice(outcomes, p=[p_away, p_draw, p_home])

    if result == 'Draw':
        return np.random.choice([team_a, team_b])
    return result

# 3. The Monte Carlo Bracket Simulator
def run_monte_carlo_tournament(iterations=10000):
    champions = []

    print(f"Running Monte Carlo Simulation ({iterations} universes)...")

    for _ in range(iterations):
        # Semi-Finals
        finalist_1 = simulate_knockout_match("Spain", "Brazil")
        finalist_2 = simulate_knockout_match("France", "Argentina")

        # The Final
        winner = simulate_knockout_match(finalist_1, finalist_2)
        champions.append(winner)

    return Counter(champions)

# 4. Execute the Simulation
results = run_monte_carlo_tournament(10000)

print("\n--- World Cup Monte Carlo Results (10,000 Simulations) ---")
total = sum(results.values())
for team, wins in results.most_common():
    win_percentage = (wins / total) * 100
    print(f"{team}: {wins} tournament wins ({win_percentage:.2f}%)")

Chapter 1

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

# 1. The Correlation Matrix
# We analyze X_upgraded to see how our engineered features interact
plt.figure(figsize=(10, 8))
correlation_matrix = X_upgraded.corr()

# Generate a clean heatmap using Seaborn
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix: Pre-Match Variables")
plt.tight_layout()
plt.show()

# 2. Principal Component Analysis (PCA)
# We apply PCA to our SCALED training data to see the mathematical variance
pca = PCA()
pca.fit(X_train_scaled_up)

# Calculate the cumulative explained variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

# Plot the PCA curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--', color='b')
plt.axhline(y=0.90, color='r', linestyle='-') # The 90% variance threshold
plt.text(1.5, 0.91, '90% Variance Threshold', color='red')

plt.title("PCA: Cumulative Explained Variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Variance")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Print the hard numbers for your thesis text
print("--- PCA Breakdown ---")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"Principal Component {i+1}: {var*100:.2f}% of variance explained")

Chapter 2

In [ ]:
from sklearn.neural_network import MLPClassifier

# 1. Initialize the Neural Network Architecture
# We'll use two hidden layers (16 neurons, then 8 neurons) with ReLU activation.
# max_iter is set to 1000 to ensure the network has time to converge.
print("Training the Deep Learning Engine (MLP)...")
mlp_model = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42
)

# 2. Fit the model to our scaled Elo + Form data
mlp_model.fit(X_train_scaled_up, y_train_up)

# 3. Evaluate the Neural Network
y_pred_mlp = mlp_model.predict(X_test_scaled_up)
print("\n--- Neural Network (MLP) Evaluation ---")
print(classification_report(y_test_up, y_pred_mlp, target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)']))

# 4. Compare the baseline accuracy of the two architectures
svm_acc = svm_model_final.score(X_test_scaled_up, y_test_up)
mlp_acc = mlp_model.score(X_test_scaled_up, y_test_up)

print(f"\nModel Showdown - SVM Accuracy: {svm_acc*100:.2f}% | MLP Accuracy: {mlp_acc*100:.2f}%")